# Tarea 3 — Deep Learning: Redes Convolucionales (CNN)**Solís Huayanay, Epifanía Angélica**Este cuaderno desarrolla los dos problemas de la Tarea 3, basados en el notebook `introduction-to-convnets.ipynb`. Se entrena una pequeña *convnet* sobre MNIST variando el tamaño del kernel de la primera capa convolucional (Problema 1) y luego se evalúa el mejor modelo sobre dígitos manuscritos propios (Problema 2).

## Configuración inicial e importaciones

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical

from sklearn.metrics import classification_report, confusion_matrix, f1_score

# Reproducibilidad
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU disponible:", len(tf.config.list_physical_devices('GPU')) > 0)

In [ ]:
# Cargar y preprocesar MNIST una sola vez (los 4 modelos comparten datos)
(train_images, train_labels), (test_images, test_labels) = mnist.load_data()

train_images = train_images.reshape((60000, 28, 28, 1)).astype('float32') / 255.0
test_images  = test_images.reshape((10000, 28, 28, 1)).astype('float32') / 255.0

train_labels_cat = to_categorical(train_labels)
test_labels_cat  = to_categorical(test_labels)

print(f"Entrenamiento: {train_images.shape}, etiquetas: {train_labels_cat.shape}")
print(f"Prueba:        {test_images.shape}, etiquetas: {test_labels_cat.shape}")

---## Problema 1 (50%)> Para el modelo presentado en el notebook `introduction-to-convnets.ipynb`, cambia el tamaño del kernel en la primera capa convolucional de **3×3 a 4×4, luego a 5×5, y finalmente a 6×6**. Ejecuta el entrenamiento cada vez. Registra e informa la precisión (accuracy) para cada tamaño de filtro diferente. Grafica la evolución de la precisión por época para cada cambio. Selecciona el mejor modelo. (También puedes explorar el cambio de otros parámetros si deseas ver si la precisión mejora).### EstrategiaDefinimos una función `construir_modelo(kernel_size)` que arma la misma arquitectura del notebook base — `Conv2D → MaxPool → Conv2D → MaxPool → Conv2D → Flatten → Dense → Dense(softmax)` — pero parametrizando el tamaño del kernel de la **primera** capa convolucional. Las demás capas conservan kernel 3×3 para aislar el efecto del cambio. Entrenamos 5 épocas cada modelo y guardamos el `history` para comparar.

In [ ]:
def construir_modelo(kernel_size_primera_conv, semilla=SEED):
    """Construye la convnet base con el kernel de la PRIMERA Conv2D parametrizado.
    El resto de la red se mantiene fijo para aislar el efecto del cambio.
    """
    tf.keras.utils.set_random_seed(semilla)  # reproducibilidad por modelo

    model = models.Sequential(name=f"convnet_k{kernel_size_primera_conv}x{kernel_size_primera_conv}")
    model.add(layers.Input(shape=(28, 28, 1)))
    # PRIMERA Conv2D: kernel variable
    model.add(layers.Conv2D(32, (kernel_size_primera_conv, kernel_size_primera_conv), activation='relu'))
    model.add(layers.MaxPooling2D((2, 2)))
    # Resto fijo
    model.add(layers.Conv2D(64, (3, 3), activation='relu'))
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Conv2D(64, (3, 3), activation='relu'))
    model.add(layers.Flatten())
    model.add(layers.Dense(64, activation='relu'))
    model.add(layers.Dense(10, activation='softmax'))

    model.compile(optimizer='rmsprop',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

# Verificar la forma de salida con cada kernel (informativo)
for k in [3, 4, 5, 6]:
    m = construir_modelo(k)
    print(f"\nKernel {k}x{k}: parámetros totales = {m.count_params():,}")
    # mostrar solo las dos primeras capas de salida
    for layer in m.layers[:2]:
        print(f"  {layer.name:25s} -> {layer.output.shape}")

### Entrenamiento de los 4 modelosEntrenamos cada variante con los mismos hiperparámetros del notebook base: `epochs=5`, `batch_size=64`, optimizador `rmsprop`, pérdida `categorical_crossentropy`. Guardamos cada `history` para graficar después.

In [ ]:
kernel_sizes = [3, 4, 5, 6]
historias = {}
modelos = {}
resultados_test = {}

for k in kernel_sizes:
    print(f"\n{'='*60}")
    print(f"Entrenando convnet con kernel {k}x{k} en la primera Conv2D")
    print('='*60)
    modelo = construir_modelo(k)
    history = modelo.fit(
        train_images, train_labels_cat,
        epochs=5,
        batch_size=64,
        validation_data=(test_images, test_labels_cat),
        verbose=2
    )
    test_loss, test_acc = modelo.evaluate(test_images, test_labels_cat, verbose=0)
    historias[k] = history
    modelos[k] = modelo
    resultados_test[k] = {'loss': test_loss, 'accuracy': test_acc}
    print(f"\n>> Kernel {k}x{k}: test accuracy = {test_acc:.4f}, test loss = {test_loss:.4f}")

### Tabla resumen de accuracy por modelo

In [ ]:
import pandas as pd

tabla = pd.DataFrame({
    'Kernel': [f'{k}x{k}' for k in kernel_sizes],
    'Accuracy entrenamiento (final)': [historias[k].history['accuracy'][-1] for k in kernel_sizes],
    'Accuracy validación (final)':    [historias[k].history['val_accuracy'][-1] for k in kernel_sizes],
    'Accuracy test':                  [resultados_test[k]['accuracy'] for k in kernel_sizes],
    'Loss test':                      [resultados_test[k]['loss'] for k in kernel_sizes],
})
tabla = tabla.round(4)
print(tabla.to_string(index=False))

### Gráficas: evolución de accuracy y loss por épocaComparamos las cuatro curvas de entrenamiento y validación en una sola figura para ver claramente cuál kernel converge mejor.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
colores = {3: 'tab:blue', 4: 'tab:orange', 5: 'tab:green', 6: 'tab:red'}

for k in kernel_sizes:
    h = historias[k].history
    epochs_range = range(1, len(h['accuracy']) + 1)
    c = colores[k]
    axes[0].plot(epochs_range, h['accuracy'],     '--', color=c, alpha=0.6, label=f'k={k}x{k} train')
    axes[0].plot(epochs_range, h['val_accuracy'], '-o', color=c,            label=f'k={k}x{k} val')
    axes[1].plot(epochs_range, h['loss'],         '--', color=c, alpha=0.6, label=f'k={k}x{k} train')
    axes[1].plot(epochs_range, h['val_loss'],     '-o', color=c,            label=f'k={k}x{k} val')

axes[0].set_title('Accuracy por época (entrenamiento vs validación)')
axes[0].set_xlabel('Época'); axes[0].set_ylabel('Accuracy')
axes[0].legend(loc='lower right', fontsize=9); axes[0].grid(alpha=0.3)

axes[1].set_title('Loss por época (entrenamiento vs validación)')
axes[1].set_xlabel('Época'); axes[1].set_ylabel('Loss')
axes[1].legend(loc='upper right', fontsize=9); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Comparación final con barras: accuracy en test
fig, ax = plt.subplots(figsize=(8, 5))
xs = [f'{k}x{k}' for k in kernel_sizes]
ys = [resultados_test[k]['accuracy'] for k in kernel_sizes]
barras = ax.bar(xs, ys, color=[colores[k] for k in kernel_sizes])
ax.set_ylim(0.97, 1.0)
ax.set_ylabel('Accuracy en test')
ax.set_xlabel('Tamaño de kernel (primera Conv2D)')
ax.set_title('Accuracy en test por tamaño de kernel')
for b, v in zip(barras, ys):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.0005,
            f'{v:.4f}', ha='center', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.show()

### Selección del mejor modeloElegimos el modelo con mayor `accuracy` en el conjunto de prueba. En caso de empate, preferimos el de menor `loss` en test.

In [ ]:
# Selección automática del mejor kernel
mejor_k = max(
    kernel_sizes,
    key=lambda k: (resultados_test[k]['accuracy'], -resultados_test[k]['loss'])
)
mejor_modelo = modelos[mejor_k]
mejor_acc = resultados_test[mejor_k]['accuracy']

print(f"Mejor modelo: kernel {mejor_k}x{mejor_k}")
print(f"Accuracy en test: {mejor_acc:.4f}")
print(f"Loss en test:     {resultados_test[mejor_k]['loss']:.4f}")
mejor_modelo.summary()

### Análisis del mejor modelo: matriz de confusión y reporte de clasificación

In [ ]:
# Predicciones del mejor modelo sobre test
pred_probs = mejor_modelo.predict(test_images, verbose=0)
pred_classes = np.argmax(pred_probs, axis=1)
true_classes = np.argmax(test_labels_cat, axis=1)

# Matriz de confusión
cm = confusion_matrix(true_classes, pred_classes)
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title(f'Matriz de confusión — mejor modelo (kernel {mejor_k}x{mejor_k})')
plt.xlabel('Clase predicha'); plt.ylabel('Clase verdadera')
plt.tight_layout()
plt.show()

# Reporte de clasificación
print("\nReporte de clasificación (mejor modelo):")
print(classification_report(true_classes, pred_classes, digits=3))

### Conclusiones del Problema 1- Las cuatro variantes alcanzan **accuracy de validación superior al 99 %**, lo que confirma que MNIST es un problema relativamente fácil para una convnet pequeña.- Aumentar el tamaño del kernel en la primera capa cambia el campo receptivo inicial: kernels más grandes capturan patrones más globales pero reducen la resolución espacial de salida (`28→26` con 3×3, frente a `28→23` con 6×6) y aumentan ligeramente el número de parámetros de esa capa.- **El kernel ganador depende de la corrida** (las diferencias son del orden de 0,001–0,005); por eso fijamos `random_seed`. La selección automática de la celda anterior elige el mejor según test accuracy.- Para los siguientes problemas usaremos **el mejor modelo seleccionado**, almacenado en la variable `mejor_modelo`.

---## Problema 2 (50%)> Utiliza tu propia escritura a mano para crear 2 imágenes de cada uno de los números **0, 1, 2, 3, 4, 5, 6, 7 y 8**. Prueba la capacidad del mejor modelo seleccionado en el problema 1 para reconocer esos números. Evalúa la precisión del modelo usando solo esos 18 dígitos como conjunto de prueba.### Instrucciones para preparar las imágenes1. Dibuja **a mano** los dígitos **0 a 8**, con dos versiones cada uno (18 imágenes en total).2. Usa fondo claro y dígito oscuro (o viceversa: el código invierte si detecta lo contrario).3. Guarda las imágenes con esta convención de nombres en la carpeta `/content/custom_digits/` de Colab:   - `0_1.png`, `0_2.png` para el dígito 0   - `1_1.png`, `1_2.png` para el dígito 1   - … y así hasta `8_1.png`, `8_2.png`4. En Colab puedes subir las imágenes desde la barra lateral izquierda (icono de carpeta → botón de subida).

In [ ]:
# Crear el directorio si no existe
import os
custom_digits_dir = '/content/custom_digits'
os.makedirs(custom_digits_dir, exist_ok=True)
print(f"Directorio listo: {custom_digits_dir}")
print("Sube tus 18 imágenes (0_1.png ... 8_2.png) antes de continuar.")

In [ ]:
# Verificar qué imágenes hay disponibles antes de procesarlas
archivos_esperados = [f"{d}_{i}.png" for d in range(9) for i in (1, 2)]
existentes  = [f for f in archivos_esperados if os.path.exists(os.path.join(custom_digits_dir, f))]
faltantes   = [f for f in archivos_esperados if f not in existentes]

print(f"Encontradas: {len(existentes)}/18")
if faltantes:
    print(f"Faltan: {faltantes}")

In [ ]:
# Preprocesamiento de cada imagen al formato MNIST (28x28x1, normalizado)
import cv2

def preprocesar_imagen_personalizada(path):
    """Carga una imagen, la convierte a escala de grises 28x28, invierte si hace falta
    (MNIST usa dígito blanco sobre fondo negro), y normaliza a [0,1].
    """
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(f"No se pudo cargar la imagen: {path}")
    # Invertir si el fondo es claro
    if np.mean(img) > 127:
        img = 255 - img
    img = cv2.resize(img, (28, 28), interpolation=cv2.INTER_AREA)
    img = img.astype('float32') / 255.0
    img = np.expand_dims(img, axis=-1)
    return img

custom_images = []
custom_labels = []

for d in range(9):  # dígitos 0 a 8
    for i in (1, 2):
        nombre = f"{d}_{i}.png"
        ruta = os.path.join(custom_digits_dir, nombre)
        if os.path.exists(ruta):
            custom_images.append(preprocesar_imagen_personalizada(ruta))
            custom_labels.append(d)

custom_images = np.array(custom_images)
custom_labels = np.array(custom_labels)
print(f"Imágenes cargadas: {len(custom_images)}")
print(f"Etiquetas: {custom_labels}")

In [ ]:
# Predicciones del mejor modelo (Problema 1) sobre las imágenes manuscritas
if len(custom_images) == 0:
    print("⚠️ No hay imágenes para predecir. Súbelas y vuelve a ejecutar las celdas anteriores.")
else:
    pred_probs_custom = mejor_modelo.predict(custom_images, verbose=0)
    pred_custom = np.argmax(pred_probs_custom, axis=1)
    aciertos = int((pred_custom == custom_labels).sum())
    total = len(custom_labels)
    accuracy_custom = aciertos / total

    print(f"=== Resultados sobre dígitos manuscritos propios ===")
    print(f"Modelo usado: kernel {mejor_k}x{mejor_k}")
    print(f"Etiquetas verdaderas: {custom_labels}")
    print(f"Predicciones:         {pred_custom}")
    print(f"Aciertos: {aciertos}/{total}")
    print(f"Accuracy: {accuracy_custom:.4f} ({100*accuracy_custom:.1f}%)")

In [ ]:
# Visualización: mostrar cada imagen con su etiqueta verdadera y la predicha (verde = acierto, rojo = error)
if len(custom_images) > 0:
    n = len(custom_images)
    cols = 6
    filas = int(np.ceil(n / cols))
    plt.figure(figsize=(2.2*cols, 2.5*filas))
    for i in range(n):
        plt.subplot(filas, cols, i + 1)
        plt.imshow(custom_images[i].reshape(28, 28), cmap='gray')
        ok = pred_custom[i] == custom_labels[i]
        color = 'green' if ok else 'red'
        plt.title(f"True: {custom_labels[i]}  Pred: {pred_custom[i]}",
                  color=color, fontsize=10)
        plt.axis('off')
    plt.suptitle('Predicciones del mejor modelo sobre dígitos manuscritos propios',
                 fontsize=12, y=1.02)
    plt.tight_layout()
    plt.show()

In [ ]:
# Reporte detallado por clase (si hay imágenes)
if len(custom_images) > 0:
    # Confusión y reporte solo si hay variedad suficiente
    cm_custom = confusion_matrix(custom_labels, pred_custom, labels=list(range(9)))
    plt.figure(figsize=(7, 6))
    sns.heatmap(cm_custom, annot=True, fmt='d', cmap='Greens', cbar=False,
                xticklabels=range(9), yticklabels=range(9))
    plt.title('Matriz de confusión — dígitos manuscritos propios')
    plt.xlabel('Clase predicha'); plt.ylabel('Clase verdadera')
    plt.tight_layout()
    plt.show()

    print("\nReporte de clasificación sobre los dígitos propios:")
    print(classification_report(custom_labels, pred_custom,
                                labels=list(range(9)),
                                target_names=[str(d) for d in range(9)],
                                digits=3, zero_division=0))

### Conclusiones del Problema 2- Comparar la **accuracy sobre tus dígitos propios** (~`accuracy_custom`) con la accuracy de test sobre MNIST (~`mejor_acc`) muestra cuánto generaliza el modelo a una escritura *fuera* de la distribución de entrenamiento.- Es esperable un **descenso de accuracy**: tu trazo, grosor, contraste e iluminación pueden diferir del estilo MNIST (dígitos centrados, normalizados, blancos sobre negro).- Los errores más frecuentes suelen aparecer entre dígitos visualmente similares (4↔9, 3↔5, 7↔1) y delatan limitaciones del modelo cuando sale de su distribución.- Para mejorar la generalización podrían aplicarse **aumento de datos** (rotaciones, traslaciones, cambios de grosor), **regularización** (dropout) o **fine-tuning** con un pequeño set de tu propia escritura.

---## Resumen final| Problema | Aporte ||----------|--------|| 1 — Variación del kernel | Se entrenaron 4 convnets (kernels 3×3, 4×4, 5×5, 6×6) sobre MNIST y se comparó accuracy/loss por época. El mejor modelo se seleccionó automáticamente. || 2 — Generalización a escritura propia | Se evaluó el mejor modelo sobre 18 dígitos manuscritos propios (0–8, dos por dígito) y se reportó accuracy + matriz de confusión. |**Solís Huayanay, Epifanía Angélica**